# Assigment 1 
* omar     2330200
* albara   2239515
* mo'men   2130462

## library

In [2]:
import pandas as pd
import os
import re
from sqlalchemy import create_engine, inspect ,text 
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
from time import sleep
import time

C:\Users\albar\AppData\Local\Temp\ipykernel_2448\275432584.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


## Q1 -- Data Extracting (Scraping)

In [5]:
# Set up Selenium WebDriver
chrome_options = Options()
chrome_options.add_argument('--headless')  # Run in headless mode (no browser window)
chrome_options.add_argument('--user-agent=Mozilla/5.0')  # Set user agent
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
 
# Function to get movie details from individual movie page
def get_movie_details(movie_url):
    driver.get(movie_url)
    sleep(7)  # Wait for the page to load
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    details = {}
    try:
        # Release Date
        release_date = soup.select_one('li[data-testid="title-details-releasedate"] a')
        details['Release Date'] = release_date.text.strip() if release_date else 'Not specified'
        # Country of Origin
        country = soup.select_one('li[data-testid="title-details-origin"] a')
        details['Country of Origin'] = country.text.strip() if country else 'Not specified'
        # Official Sites
        official_site = soup.select_one('li[data-testid="title-details-officialsites"] div.ipc-metadata-list-item__content-container a[href*="offsite"]')
        details['Official Sites'] = official_site['href'] if official_site else 'Not specified'
        # Languages
        languages = [lang.text.strip() for lang in soup.select('li[data-testid="title-details-languages"] div.ipc-metadata-list-item__content-container a')]
        details['Languages'] = ', '.join(languages) if languages else 'Not specified'
        # Filming Locations
        filming_locations = soup.select_one('li[data-testid="title-details-filminglocations"] div.ipc-metadata-list-item__content-container a')
        details['Filming Locations'] = filming_locations.text.strip() if filming_locations else 'Not specified'
        # Production Companies
        prod_companies = [comp.text.strip() for comp in soup.select('li[data-testid="title-details-companies"] div.ipc-metadata-list-item__content-container a')]
        details['Production Companies'] = ', '.join(prod_companies) if prod_companies else 'Not specified'
        # Budget
        budget = soup.select_one('li[data-testid="title-boxoffice-budget"] div.ipc-metadata-list-item__content-container')
        details['Budget'] = budget.text.strip() if budget else 'Unknown'
        # Gross Worldwide Revenue
        gross = soup.select_one('li[data-testid="title-boxoffice-cumulativeworldwidegross"] div.ipc-metadata-list-item__content-container')
        details['Gross Worldwide Revenue'] = gross.text.strip() if gross else 'Unknown'
        # Director(s)
        director = soup.select_one('li[data-testid="title-pc-principal-credit"] a')
        details['Director(s)'] = director.text.strip() if director else 'Not specified'
        # Genre(s)
        genres = [genre.text.strip() for genre in soup.select('span.ipc-chip__text')]
        details['Genre(s)'] = ', '.join(genres) if genres else 'Not specified'
    except Exception as e:
        print(f"Error extracting details for {movie_url}: {e}")
    return details
 
# Main scraping function for movies only
def scrape_imdb_top_250_movies():
    url = 'https://www.imdb.com/chart/top/'
    driver.get(url)
    sleep(7)  # Wait for the page to load
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    movies_data = []
    # Get movie list
    movie_list = soup.select('.ipc-metadata-list-summary-item')
    for idx, movie in enumerate(movie_list[:5], 1):  # Limit to 250
        title_link = movie.select_one('.ipc-title__text')
        if not title_link:
            continue
        title = title_link.text.strip().split('. ')[1]
        movie_url = 'https://www.imdb.com' + title_link.find_parent('a')['href']
        # Basic info
        rating_elem = movie.select_one('span.ipc-rating-star--rating')
        votes_elem = movie.select_one('span.ipc-rating-star--voteCount')
        rating = float(rating_elem.text.strip()) if rating_elem and rating_elem.text.strip().replace('.', '').isdigit() else 0.0
        votes = votes_elem.text.strip().lstrip('(').rstrip(')') if votes_elem else '0'
        year_elem = movie.select_one('.cli-title-metadata-item')
        year = year_elem.text.strip() if year_elem else 'Unknown'
        movie_data = {
            'Movie_ID': idx,
            'Title': title,
            'Release Year': year,
            'Rating': rating,
            'Votes': votes,
            'Movie_URL': movie_url  # Add the movie URL to the data
        }
        # Get detailed info
        details = get_movie_details(movie_url)
        movie_data.update(details)
        movies_data.append(movie_data)
        print(f"Scraped movie {idx}: {title}")
        sleep(1)  # Be polite to the server
    # Create dataframe
    movies_df = pd.DataFrame(movies_data)
    return movies_df
 
# Execute scraping for movies
try:
    movies_df = scrape_imdb_top_250_movies()
    # Add error handling for file saving
    try:
        movies_df.to_csv('movies1.csv', index=False)
    except PermissionError as e:
        print(f"PermissionError: {e}")
        print("Trying to save to a different location...")
        movies_df.to_csv('C:/Users/DELL/Desktop/movies.csv', index=False)
    print("Movies scraping completed!")
    print(movies_df.head())
finally:
    driver.quit()  # Ensure the driver is closed

Scraped movie 1: The Shawshank Redemption
Scraped movie 2: The Godfather
Scraped movie 3: The Dark Knight
Scraped movie 4: The Godfather Part II
Scraped movie 5: 12 Angry Men
Movies scraping completed!
   Movie_ID                     Title Release Year  Rating Votes  \
0         1  The Shawshank Redemption         1994     9.3    3M   
1         2             The Godfather         1972     9.2  2.1M   
2         3           The Dark Knight         2008     9.0    3M   
3         4     The Godfather Part II         1974     9.0  1.4M   
4         5              12 Angry Men         1957     9.0  921K   

                                           Movie_URL  Release Date  \
0  https://www.imdb.com/title/tt0111161/?ref_=cht...  Release date   
1  https://www.imdb.com/title/tt0068646/?ref_=cht...  Release date   
2  https://www.imdb.com/title/tt0468569/?ref_=cht...  Release date   
3  https://www.imdb.com/title/tt0071562/?ref_=cht...  Release date   
4  https://www.imdb.com/title/tt0050083

In [7]:
# Load the movies data
try:
    movies_df = pd.read_csv('movies1.csv')
    print("Successfully loaded movies.csv")
except Exception as e:
    print(f"Error loading movies.csv: {e}")
    raise

# Check if Movie_URL column exists
if 'Movie_URL' not in movies_df.columns:
    raise ValueError("Movie_URL column not found in movies.csv")

# Prepare a list to store cast and crew data
cast_crew_data = []

# Set up Selenium WebDriver
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

# Iterate over each movie
for index, row in movies_df.iterrows():
    movie_id = row['Movie_ID']
    movie_title = row['Title']
    movie_url = row['Movie_URL']
    
    print(f"\nProcessing movie: {movie_title} (ID: {movie_id})")
    print(f"URL: {movie_url}")
    
    try:
        # Use Selenium to load the page
        driver.get(movie_url)
        time.sleep(3)  # Wait for the page to load fully
        
        # Parse the page content
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        
        # Print the page title to verify we're on the correct page
        page_title = soup.find('title').get_text(strip=True) if soup.find('title') else 'No title found'
        print(f"Page title: {page_title}")
        
        # Find the cast section using data-testid (adjust based on inspection)
        cast_section = soup.find('div', attrs={'data-testid': 'title-cast'})
        
        if cast_section:
            print(f"Found cast section for {movie_title}")
            # Find all cast items (adjust class based on inspection)
            cast_items = cast_section.find_all('div', class_='sc-bfec09a1-5')
            
            if not cast_items:
                # Try a more general approach
                cast_items = cast_section.find_all('div', recursive=True)
            
            # Use a set to avoid duplicates
            seen_names = set()
            
            for cast_item in cast_items:
                # Look for the actor's name
                name_tag = cast_item.find('a', href=lambda x: x and '/name/' in x)
                name = name_tag.get_text(strip=True) if name_tag else None
                
                if not name:
                    continue
                
                # Avoid duplicates
                if name in seen_names:
                    continue
                seen_names.add(name)
                
                # Look for the character name
                character_tag = cast_item.find('div', class_=lambda x: x and 'character' in x.lower())
                if not character_tag:
                    character_tag = cast_item.find('span', class_=lambda x: x and 'character' in x.lower())
                character = character_tag.get_text(strip=True) if character_tag else ''
                
                print(f"Found cast member: {name} as {character}")
                
                # Append to the data list
                cast_crew_data.append({
                    'Movie_ID': movie_id,
                    'Name': name,
                    'Role_Type': 'Actor',
                    'Character_Position': character
                })
        else:
            print(f"No cast section found for {movie_title}")
            # Fallback: Search for a section with "Cast" in the heading
            for section in soup.find_all(['section', 'div']):
                heading = section.find(['h2', 'h3'])
                if heading and 'cast' in heading.get_text(strip=True).lower():
                    cast_section = section
                    break
            
            if cast_section:
                print(f"Found cast section (fallback method) for {movie_title}")
                cast_items = cast_section.find_all('div', recursive=True)
                
                seen_names = set()
                
                for cast_item in cast_items:
                    name_tag = cast_item.find('a', href=lambda x: x and '/name/' in x)
                    name = name_tag.get_text(strip=True) if name_tag else None
                    
                    if not name:
                        continue
                    
                    if name in seen_names:
                        continue
                    seen_names.add(name)
                    
                    character_tag = cast_item.find('div', class_=lambda x: x and 'character' in x.lower())
                    if not character_tag:
                        character_tag = cast_item.find('span', class_=lambda x: x and 'character' in x.lower())
                    character = character_tag.get_text(strip=True) if character_tag else ''
                    
                    print(f"Found cast member (fallback): {name} as {character}")
                    
                    cast_crew_data.append({
                        'Movie_ID': movie_id,
                        'Name': name,
                        'Role_Type': 'Actor',
                        'Character_Position': character
                    })
            else:
                print(f"Could not find cast section even with fallback method for {movie_title}")
                print(f"Page HTML snippet: {soup.prettify()[:500]}...")
        
        # Add a delay to avoid overwhelming the server
        time.sleep(2)
    
    except Exception as e:
        print(f"Error fetching data for {movie_title}: {e}")
        continue

# Close the browser
driver.quit()

# Convert the data to a DataFrame
cast_crew_df = pd.DataFrame(cast_crew_data)

# Check if the DataFrame is empty
if cast_crew_df.empty:
    print("No cast data collected. The DataFrame is empty.")
else:
    print(f"Collected data for {len(cast_crew_df)} cast members.")

# Save to CSV
cast_crew_df.to_csv('full_cast_and_crew.csv', index=False)
print("Saved cast and crew data to 'full_cast_and_crew.csv'")

Successfully loaded movies.csv

Processing movie: The Shawshank Redemption (ID: 1)
URL: https://www.imdb.com/title/tt0111161/?ref_=chttp_t_1
Page title: The Shawshank Redemption (1994) - IMDb
No cast section found for The Shawshank Redemption
Found cast section (fallback method) for The Shawshank Redemption
Found cast member (fallback): Frank Darabont as Andy Dufresne
Found cast member (fallback): Stephen King as 
Found cast member (fallback): Tim Robbins as 
Found cast member (fallback): Morgan Freeman as Ellis Boyd 'Red' Redding
Found cast member (fallback): Bob Gunton as Warden Norton
Found cast member (fallback): William Sadler as Heywood
Found cast member (fallback): Clancy Brown as Captain Hadley
Found cast member (fallback): Gil Bellows as Tommy
Found cast member (fallback): Mark Rolston as Bogs Diamond
Found cast member (fallback): James Whitmore as Brooks Hatlen
Found cast member (fallback): Jeffrey DeMunn as 1946 D.A.
Found cast member (fallback): Larry Brandenburg as Skeet
F

## Q2 -- Data Preprocessing and Transformation

In [8]:
# Load the datasets
movies_df = pd.read_csv('movies1.csv')
cast_crew_df = pd.read_csv('full_cast_and_crew.csv')

# --- Step 1: Clean the Movies Data ---

# 1.1 Handle 'Votes' column: Convert '3M' to 3000000, '919K' to 919000, etc.
def convert_votes(votes):
    if isinstance(votes, str):
        if 'M' in votes:
            return int(float(votes.replace('M', '')) * 1_000_000)
        elif 'K' in votes:
            return int(float(votes.replace('K', '')) * 1_000)
    return int(votes)

movies_df['Votes'] = movies_df['Votes'].apply(convert_votes)

# 1.2 Clean 'Budget' column: Remove '$', ' (estimated)', and commas, then convert to numeric
def clean_budget(budget):
    if pd.isna(budget):
        return None
    
    # Store the original budget for debugging
    original_budget = budget
    print(f"Original budget: {original_budget}")
    
    # Remove '(estimated)' first
    budget = budget.replace(' (estimated)', '')
    print(f"After removing '(estimated)': {budget}")
    
    # Remove currency symbols and commas
    budget = budget.replace('$', '').replace('¥', '').replace('R$', '').replace(',', '')
    print(f"After removing currency symbols and commas: {budget}")
    
    # Convert to integer
    try:
        budget_value = int(budget)
    except ValueError as e:
        print(f"Error converting budget to int: {budget}, Error: {e}")
        return None
    
    # Apply currency conversion if needed
    if '¥' in original_budget:
        # Convert Yen to USD (approximate historical rate: ¥125 = $1 in 1954)
        print(f"Converting Yen to USD: {budget_value} / 125")
        return budget_value / 125
    elif 'R$' in original_budget:
        # Convert Brazilian Real to USD (approximate rate: R$3.3 = $1 in 2002)
        print(f"Converting BRL to USD: {budget_value} / 3.3")
        return budget_value / 3.3
    
    return budget_value

movies_df['Budget'] = movies_df['Budget'].apply(clean_budget)

# 1.3 Clean 'Gross Worldwide Revenue' column: Remove '$' and commas, then convert to numeric
def clean_gross(gross):
    if pd.isna(gross):
        return None
    return int(gross.replace('$', '').replace(',', ''))

movies_df['Gross Worldwide Revenue'] = movies_df['Gross Worldwide Revenue'].apply(clean_gross)

# 1.4 Clean 'Languages' and 'Genre(s)' columns: Split into lists and remove 'Back to top'
movies_df['Languages'] = movies_df['Languages'].str.split(', ').apply(lambda x: [lang.strip() for lang in x])
movies_df['Genre(s)'] = movies_df['Genre(s)'].str.replace('Back to top', '').str.split(', ').apply(lambda x: [genre.strip() for genre in x if genre.strip()])

# 1.5 Clean 'Director(s)' column: Split into list if multiple directors
movies_df['Director(s)'] = movies_df['Director(s)'].str.split(', ').apply(lambda x: [director.strip() for director in x])

# 1.6 Clean 'Production Companies' column: Split into list
movies_df['Production Companies'] = movies_df['Production Companies'].str.split(', ').apply(lambda x: [company.strip() for company in x])

# 1.7 Drop unnecessary columns: 'Release Date', 'Official Sites' (all 'Not specified'), 'Movie_URL'
movies_df = movies_df.drop(columns=['Release Date', 'Official Sites', 'Movie_URL'])

# 1.8 Handle missing values: Check for any critical missing data
print("Missing values in movies_df:")
print(movies_df.isnull().sum())

# --- Step 2: Clean the Cast and Crew Data ---

# 2.1 Clean 'Character_Position' column: Remove extra spaces, newlines, and '(as ...)' notes
def clean_character_position(position):
    if pd.isna(position):
        return ''
    # Remove extra spaces and newlines
    position = ' '.join(position.split())
    # Remove '(as ...)' notes
    position = re.sub(r'\(as [^\)]+\)', '', position).strip()
    return position

cast_crew_df['Character_Position'] = cast_crew_df['Character_Position'].apply(clean_character_position)

# 2.2 Clean 'Name' column: Remove extra spaces
cast_crew_df['Name'] = cast_crew_df['Name'].str.strip()

# 2.3 Handle missing values in cast_crew_df
print("\nMissing values in cast_crew_df:")
print(cast_crew_df.isnull().sum())

# 2.4 Remove duplicates in cast_crew_df (if any)
cast_crew_df = cast_crew_df.drop_duplicates()

# --- Step 3: Merge the Datasets (Optional) ---
# We can merge the datasets on 'Movie_ID' if needed for analysis
merged_df = pd.merge(movies_df, cast_crew_df, on='Movie_ID', how='left')

# --- Step 4: Save the Cleaned Data ---
# Save the cleaned movies data
movies_df.to_csv('cleaned_movies.csv', index=False)

# Save the cleaned cast and crew data
cast_crew_df.to_csv('cleaned_cast_and_crew.csv', index=False)

# Save the merged data (optional)
merged_df.to_csv('merged_movies_cast.csv', index=False)

# --- Step 5: Display a Preview of the Cleaned Data ---
print("\nPreview of cleaned movies_df:")
print(movies_df.head())
print("\nPreview of cleaned cast_crew_df:")
print(cast_crew_df.head())

Original budget: $25,000,000 (estimated)
After removing '(estimated)': $25,000,000
After removing currency symbols and commas: 25000000
Original budget: $6,000,000 (estimated)
After removing '(estimated)': $6,000,000
After removing currency symbols and commas: 6000000
Original budget: $185,000,000 (estimated)
After removing '(estimated)': $185,000,000
After removing currency symbols and commas: 185000000
Original budget: $13,000,000 (estimated)
After removing '(estimated)': $13,000,000
After removing currency symbols and commas: 13000000
Original budget: $350,000 (estimated)
After removing '(estimated)': $350,000
After removing currency symbols and commas: 350000
Missing values in movies_df:
Movie_ID                   0
Title                      0
Release Year               0
Rating                     0
Votes                      0
Country of Origin          0
Languages                  0
Filming Locations          0
Production Companies       0
Budget                     0
Gross Wor

## Q3 -- Data Preprocessing and Transformation

In [ ]:
# convert form csv to sqllit
csv_file = "merged_movies_cast.csv"
df = pd.read_csv(csv_file)

sqlite_file = "movies_cast.db"
engine = create_engine(f"sqlite:///{sqlite_file}")

df.to_sql("movies_cast", engine, if_exists="replace", index=False)
 
inspector = inspect(engine)
print(df.columns)

Index(['Movie_ID', 'Title', 'Release Year', 'Rating', 'Votes',
       'Country of Origin', 'Languages', 'Filming Locations',
       'Production Companies', 'Budget', 'Gross Worldwide Revenue',
       'Director(s)', 'Genre(s)', 'Name', 'Role_Type', 'Character_Position'],
      dtype='object')


In [4]:
with engine.connect() as con:
    result = con.execute(text("""
        SELECT Title, Rating 
        FROM movies_cast 
        ORDER BY Rating DESC                  

    """))#LIMIT 10 not working 

df = pd.DataFrame(result.fetchall(), columns=result.keys())
df = df.drop_duplicates()
df.head(10)


,Title,Rating
0,The Shawshank Redemption,9.3
23,The Godfather,9.2
45,The Dark Knight,9.0
67,The Godfather Part II,9.0
88,12 Angry Men,9.0


In [14]:
with engine.connect() as con:
    result = con.execute(text("""
        SELECT "Release Year", COUNT(DISTINCT Title) AS Movie_Count
        FROM movies_cast
        GROUP BY "Release Year"
        ORDER BY Movie_Count DESC
    """))
    
df = pd.DataFrame(result.fetchall(), columns=result.keys())
df = df.drop_duplicates()

df

,Release Year,Movie_Count
0,2008,1
1,1994,1
2,1974,1
3,1972,1
4,1957,1


In [13]:
with engine.connect() as con:
    result = con.execute(text("""
        SELECT Name, COUNT(*) as movies_count
        FROM movies_cast
        GROUP BY Name
        ORDER BY movies_count DESC

    """))
    
df = pd.DataFrame(result.fetchall(), columns=result.keys())
df = df.drop_duplicates()

df.head(10)


,Name,movies_count
0,Talia Shire,2
1,Rudy Bond,2
2,Robert Duvall,2
3,Morgana King,2
4,Morgan Freeman,2
5,John Cazale,2
6,Francis Ford Coppola,2
7,Diane Keaton,2
8,Al Pacino,2
9,William Sadler,1
